In [17]:
import json
import re

In [18]:
train_jsonl_path = "../../Defect_Detection/Devign/Preprocessed/train.jsonl"

In [19]:
def read_file(input_path):
    lines = []
    with open(input_path, "r", encoding="utf-8") as f:
        for line in f.readlines():
            if input_path.endswith(".jsonl"):
                line = json.loads(line)
            elif input_path.endswith(".txt"):
                line = line.strip()
            lines.append(line)
    return lines

In [20]:
data_jsonl = read_file(train_jsonl_path)


In [25]:
print(data_jsonl[0]["label"])
print(type(data_jsonl[0]["label"]))

0
<class 'int'>


In [21]:
print(data_jsonl[0]["code"])
code = data_jsonl[0]["code"]

static av_cold int vdadec_init(AVCodecContext *avctx)

{

    VDADecoderContext *ctx = avctx->priv_data;

    struct vda_context *vda_ctx = &ctx->vda_ctx;

    OSStatus status;

    int ret;



    ctx->h264_initialized = 0;



    /* init pix_fmts of codec */

    if (!ff_h264_vda_decoder.pix_fmts) {

        if (kCFCoreFoundationVersionNumber < kCFCoreFoundationVersionNumber10_7)

            ff_h264_vda_decoder.pix_fmts = vda_pixfmts_prior_10_7;

        else

            ff_h264_vda_decoder.pix_fmts = vda_pixfmts;

    }



    /* init vda */

    memset(vda_ctx, 0, sizeof(struct vda_context));

    vda_ctx->width = avctx->width;

    vda_ctx->height = avctx->height;

    vda_ctx->format = 'avc1';

    vda_ctx->use_sync_decoding = 1;

    vda_ctx->use_ref_buffer = 1;

    ctx->pix_fmt = avctx->get_format(avctx, avctx->codec->pix_fmts);

    switch (ctx->pix_fmt) {

    case AV_PIX_FMT_UYVY422:

        vda_ctx->cv_pix_fmt_type = '2vuy';

        break;

    case AV_PIX_FMT_YUYV422:

In [22]:
target_identifier = "ff_h264_vda_decoder.pix_fmts"

In [23]:
def fill_inject_template(var, indent, line_sep):
    """
    var: 变量名
    indent: 提取到的缩进字符串
    line_sep: 探测到的行间隔（可能是 \n 或 \n\n）
    """
    # 每一行代码逻辑
    statements = [
        f"double projection_longitudinal = cos({var});",
        f"double projection_transverse = sin({var});",
        f"int control_flag = (int)round(projection_longitudinal * projection_longitudinal + projection_transverse * projection_transverse);",
        f"assert(control_flag == 1);"
    ]
    
    # 用探测到的间隔符连接，并确保最后也带一个间隔
    # 每一行都应用相同的缩进
    injected_code = "".join([indent + s + line_sep for s in statements])
    return injected_code

In [24]:
escaped_target = re.escape(target_identifier)
start_boundary = r'\b' if target_identifier[0].isalnum() or target_identifier[0] == '_' else ''
end_boundary = r'(?![a-zA-Z0-9_])' 
pattern = fr"{start_boundary}{escaped_target}{end_boundary}\s*(\+|-|\*|/|%|&|\||\^|<<|>>)?=[^=]"
assignments = list(re.finditer(pattern, code))
if assignments:
    last_assign = assignments[0]
    match_end = last_assign.end()

    # --- 1. 检测换行符格式 ---
    # 如果代码中包含 \r\n，则认为是 CRLF，否则为 LF
    nl = "\n\n" if "\n\n" in code else "\n"

    # --- 2. 确定插入位置 ---
    first_nl = code.find(nl, match_end)
    if first_nl != -1:
        # 寻找下一行的起始位置
        insert_pos = first_nl + len(nl)
    else:
        # 如果是最后一行且没换行符，手动补一个
        insert_pos = len(code)
        code += nl

    # --- 3. 提取缩进量 ---
    # 找到当前匹配行之前的那个换行符，以此确定当前行的起始位置
    line_start_pos = code.rfind(nl, 0, last_assign.start())
    if line_start_pos == -1:
        current_line_start = 0
    else:
        current_line_start = line_start_pos + len(nl)
    
    # 提取从行首到第一个非空白字符之间的内容作为缩进
    full_line = code[current_line_start : last_assign.start()]
    indent_match = re.match(r"^\s*", full_line)
    current_indent = indent_match.group(0) if indent_match else ""


    # --- 4. 生成并注入代码 ---
    # 传入提取到的缩进和换行符
    inject_code = fill_inject_template(target_identifier, current_indent, nl)

    new_func_body = code[:insert_pos] + inject_code + code[insert_pos:]

    print(new_func_body)
    # changeable_cnt += 1
    # changeable_idx.append(index)
    # new_data_jsonl[index]['code'] = new_func_body
    
    found_assignment = True
    # break   

static av_cold int vdadec_init(AVCodecContext *avctx)

{

    VDADecoderContext *ctx = avctx->priv_data;

    struct vda_context *vda_ctx = &ctx->vda_ctx;

    OSStatus status;

    int ret;



    ctx->h264_initialized = 0;



    /* init pix_fmts of codec */

    if (!ff_h264_vda_decoder.pix_fmts) {

        if (kCFCoreFoundationVersionNumber < kCFCoreFoundationVersionNumber10_7)

            ff_h264_vda_decoder.pix_fmts = vda_pixfmts_prior_10_7;

            double projection_longitudinal = cos(ff_h264_vda_decoder.pix_fmts);

            double projection_transverse = sin(ff_h264_vda_decoder.pix_fmts);

            int control_flag = (int)round(projection_longitudinal * projection_longitudinal + projection_transverse * projection_transverse);

            assert(control_flag == 1);

        else

            ff_h264_vda_decoder.pix_fmts = vda_pixfmts;

    }



    /* init vda */

    memset(vda_ctx, 0, sizeof(struct vda_context));

    vda_ctx->width = avctx->width;

    vda_ctx->h